# BACI Russia Lite Extractor

원본 `BACI_HS12_V202601.zip`을 한 번만 읽어 러시아 연구용 경량 Parquet 파일을 생성합니다.

입력 경로 자동 탐색:
- `/content/BACI`
- `/content/drive/MyDrive/BACI`

출력:
- 러시아 수입 partner×HS6×year
- 러시아 수출 partner×HS6×year
- 러시아 수입 HS6×year summary
- 러시아 수출 HS6×year summary
- 중국 world trade year summary
- `Russia_BACI_Lite_Dataset.zip`


In [ ]:
!pip -q install pandas pyarrow tqdm

from pathlib import Path
import pandas as pd, numpy as np
import zipfile, shutil, json
from tqdm.auto import tqdm

ROOT = Path("/content/russia_baci_lite")
OUT = ROOT / "output"
OUT.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [Path("/content/BACI"), Path("/content/drive/MyDrive/BACI")]

def find_baci():
    for d in SEARCH_DIRS:
        if d.exists():
            for z in d.glob("*.zip"):
                if "BACI_HS12" in z.name.upper():
                    return z
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Drive mount:", e)
    for d in SEARCH_DIRS:
        if d.exists():
            for z in d.glob("*.zip"):
                if "BACI_HS12" in z.name.upper():
                    return z
    return None

BACI_ZIP = find_baci()
if BACI_ZIP is None:
    raise FileNotFoundError("BACI_HS12_V202601.zip을 /content/BACI 또는 /content/drive/MyDrive/BACI 에 두세요.")

print("BACI:", BACI_ZIP)
print("Size:", round(BACI_ZIP.stat().st_size/1024**3, 3), "GB")


In [ ]:
BACI_VERSION = "202601"
YEARS = list(range(2015, 2025))

RUS, CHN, IND, TUR, KAZ = 643, 156, 699, 792, 398
EU27 = [40,56,100,191,196,203,208,233,246,250,276,300,348,372,
        380,428,440,442,470,528,616,620,642,703,705,724,752]

with zipfile.ZipFile(BACI_ZIP) as z:
    names = z.namelist()

annual_files = {}
for y in YEARS:
    expected = f"BACI_HS12_Y{y}_V{BACI_VERSION}.csv"
    matches = [n for n in names if Path(n).name == expected]
    if not matches:
        raise FileNotFoundError(expected)
    annual_files[y] = matches[0]

print("Validated", len(annual_files), "annual files")


In [ ]:
import_rows, export_rows, china_world_rows = [], [], []

with zipfile.ZipFile(BACI_ZIP) as z:
    for y in YEARS:
        imp_acc, exp_acc = {}, {}
        chn_world_exp, chn_world_imp = 0.0, 0.0

        with z.open(annual_files[y]) as f:
            reader = pd.read_csv(
                f, usecols=["i","j","k","v","q"], dtype={"k":"string"},
                chunksize=700_000, low_memory=False
            )
            for chunk in tqdm(reader, desc=f"Extract {y}", leave=False):
                chunk["k"] = chunk["k"].astype("string").str.zfill(6)
                chunk["v"] = pd.to_numeric(chunk["v"], errors="coerce").fillna(0.0)
                chunk["q"] = pd.to_numeric(chunk["q"], errors="coerce").fillna(0.0)
                chunk["i"] = pd.to_numeric(chunk["i"], errors="coerce")
                chunk["j"] = pd.to_numeric(chunk["j"], errors="coerce")

                imp = chunk[chunk["j"].eq(RUS)]
                if not imp.empty:
                    g = imp.groupby(["i","k"], as_index=False)[["v","q"]].sum()
                    for r in g.itertuples(index=False):
                        key = (int(r.i), str(r.k))
                        rec = imp_acc.setdefault(key, {"v":0.0,"q":0.0})
                        rec["v"] += float(r.v); rec["q"] += float(r.q)

                exp = chunk[chunk["i"].eq(RUS)]
                if not exp.empty:
                    g = exp.groupby(["j","k"], as_index=False)[["v","q"]].sum()
                    for r in g.itertuples(index=False):
                        key = (int(r.j), str(r.k))
                        rec = exp_acc.setdefault(key, {"v":0.0,"q":0.0})
                        rec["v"] += float(r.v); rec["q"] += float(r.q)

                chn_world_exp += float(chunk.loc[chunk["i"].eq(CHN), "v"].sum())
                chn_world_imp += float(chunk.loc[chunk["j"].eq(CHN), "v"].sum())

        for (partner, hs6), vals in imp_acc.items():
            import_rows.append({"year":y,"partner":partner,"hs6":hs6,
                                "value_kUSD":vals["v"],"quantity_ton":vals["q"]})
        for (partner, hs6), vals in exp_acc.items():
            export_rows.append({"year":y,"partner":partner,"hs6":hs6,
                                "value_kUSD":vals["v"],"quantity_ton":vals["q"]})
        china_world_rows.append({"year":y,
                                 "china_world_export_kUSD":chn_world_exp,
                                 "china_world_import_kUSD":chn_world_imp})

imp_detail = pd.DataFrame(import_rows)
exp_detail = pd.DataFrame(export_rows)
china_world = pd.DataFrame(china_world_rows)

for df in [imp_detail, exp_detail]:
    df["hs6"] = df["hs6"].astype("string").str.zfill(6)
    df["hs2"] = df["hs6"].str[:2]
    df["hs4"] = df["hs6"].str[:4]

print("Import detail", imp_detail.shape)
print("Export detail", exp_detail.shape)


In [ ]:
def add_partner_values(base, detail, side):
    partner_map = [(CHN,"china"),(IND,"india"),(TUR,"turkiye"),(KAZ,"kazakhstan")]
    for code,name in partner_map:
        g = detail[detail["partner"].eq(code)].groupby(["year","hs6"],as_index=False)["value_kUSD"].sum()
        g = g.rename(columns={"value_kUSD":f"{name}_{side}_kUSD"})
        base = base.merge(g,on=["year","hs6"],how="left")
    eu = detail[detail["partner"].isin(EU27)].groupby(["year","hs6"],as_index=False)["value_kUSD"].sum()
    eu = eu.rename(columns={"value_kUSD":f"eu27_{side}_kUSD"})
    base = base.merge(eu,on=["year","hs6"],how="left")
    return base

imp_summary = imp_detail.groupby(["year","hs6","hs2","hs4"],as_index=False).agg(
    world_import_kUSD=("value_kUSD","sum"),
    world_import_qty=("quantity_ton","sum")
)
imp_summary = add_partner_values(imp_summary, imp_detail, "import")

for c in ["china_import_kUSD","india_import_kUSD","turkiye_import_kUSD","kazakhstan_import_kUSD","eu27_import_kUSD"]:
    imp_summary[c] = imp_summary[c].fillna(0.0)

for name in ["china","india","turkiye","eu27"]:
    imp_summary[f"{name}_share"] = np.where(
        imp_summary["world_import_kUSD"]>0,
        imp_summary[f"{name}_import_kUSD"]/imp_summary["world_import_kUSD"],
        np.nan
    )

exp_summary = exp_detail.groupby(["year","hs6","hs2","hs4"],as_index=False).agg(
    world_export_kUSD=("value_kUSD","sum"),
    world_export_qty=("quantity_ton","sum")
)
exp_summary = add_partner_values(exp_summary, exp_detail, "export")

for c in ["china_export_kUSD","india_export_kUSD","turkiye_export_kUSD","kazakhstan_export_kUSD","eu27_export_kUSD"]:
    exp_summary[c] = exp_summary[c].fillna(0.0)

for name in ["china","india","turkiye","eu27"]:
    exp_summary[f"{name}_share"] = np.where(
        exp_summary["world_export_kUSD"]>0,
        exp_summary[f"{name}_export_kUSD"]/exp_summary["world_export_kUSD"],
        np.nan
    )

display(imp_summary.head())
display(exp_summary.head())


In [ ]:
paths = {
    "import_detail": OUT/"russia_import_partner_hs6_year.parquet",
    "export_detail": OUT/"russia_export_partner_hs6_year.parquet",
    "import_summary": OUT/"russia_import_hs6_year_summary.parquet",
    "export_summary": OUT/"russia_export_hs6_year_summary.parquet",
    "china_world": OUT/"china_world_trade_year.parquet"
}

imp_detail.to_parquet(paths["import_detail"],index=False)
exp_detail.to_parquet(paths["export_detail"],index=False)
imp_summary.to_parquet(paths["import_summary"],index=False)
exp_summary.to_parquet(paths["export_summary"],index=False)
china_world.to_parquet(paths["china_world"],index=False)

meta = {
    "source":"CEPII BACI HS2012 V202601",
    "years":YEARS,
    "files":{k:v.name for k,v in paths.items()}
}
with open(OUT/"metadata.json","w",encoding="utf-8") as f:
    json.dump(meta,f,indent=2,ensure_ascii=False)

for k,p in paths.items():
    print(k, round(p.stat().st_size/1024**2,2), "MB")

zip_out = ROOT/"Russia_BACI_Lite_Dataset.zip"
if zip_out.exists(): zip_out.unlink()
shutil.make_archive(str(zip_out.with_suffix("")),"zip",OUT)
print("ZIP:", zip_out, round(zip_out.stat().st_size/1024**2,2), "MB")

try:
    from google.colab import files
    files.download(str(zip_out))
except Exception:
    pass
